# Load_Source_Data

Loads the sample data for the **Contoso CU + Fabrikam CU merger demo** into three lakehouses in the current workspace:

| Lakehouse | Contents |
|---|---|
| `Contoso_CU` | Legacy Contoso Credit Union tables |
| `Fabrikam_CU` | Legacy Fabrikam Credit Union tables |
| `Enterprise_Ref` | Certified KPI glossary and the row-level security scope |

The notebook creates any lakehouse that does not exist, writes every table with an explicit schema, and refreshes each lakehouse's SQL analytics endpoint so the warehouse can query the new tables right away.

**Before you run it:** set `TENANT_DOMAIN` below to your tenant's domain (for example `contoso.onmicrosoft.com`). The demo personas (`west_director`, `east_director`, ...) are loaded into the RLS table with that domain.

**Data source options**
- `DATA_SOURCE = "github"` reads the CSVs straight from your copy of the repository (`REPO_RAW_BASE`).
- `DATA_SOURCE = "files"` reads them from the `Files/data` folder of the `Enterprise_Ref` lakehouse. Upload the repository's `data` folder there first (keep the `contoso_cu`, `fabrikam_cu`, `enterprise_ref` subfolders).

If you mirror the legacy sources from SQL databases instead (see `docs/04-Optional-Mirroring-and-Snowflake.md`), set `LOAD_LEGACY_SOURCES = False`. Only `Enterprise_Ref` is loaded.

In [ ]:
# ---------------- Parameters ----------------
TENANT_DOMAIN       = "yourtenant.onmicrosoft.com"   # REQUIRED: domain of the demo user accounts
DATA_SOURCE         = "github"                       # "github" or "files"
REPO_RAW_BASE       = "https://raw.githubusercontent.com/<github-user>/<repo-name>/main/data"
LOAD_LEGACY_SOURCES = True                           # False if Contoso_CU / Fabrikam_CU are mirrored databases

LAKEHOUSES = {"contoso_cu": "Contoso_CU", "fabrikam_cu": "Fabrikam_CU", "enterprise_ref": "Enterprise_Ref"}

## Table schemas

In [ ]:
from pyspark.sql import functions as F

# column -> Spark SQL type. Every CSV is read as text, then cast to these types.
SCHEMAS = {
    "dim_branch":                {"BranchId": "string", "BranchName": "string", "Region": "string", "State": "string"},
    "dim_month":                 {"MonthStart": "date"},
    "dim_member_segment":        {"SegmentId": "string", "SegmentName": "string"},
    "fact_monthly_kpi":          {"MonthStart": "date", "BranchId": "string", "Region": "string", "LoansBooked": "int",
                                  "DepositGrowthPct": "double", "NPS": "double", "DelinquencyRatePct": "double",
                                  "DigitalAdoptionPct": "double"},
    "fact_driver_breakdown":     {"MonthStart": "date", "BranchId": "string", "DriverName": "string",
                                  "ImpactScore": "double", "DriverValue": "double"},
    "glossary_kpi_definitions":  {"KPI": "string", "Definition": "string", "Owner": "string", "Calculation": "string"},
    "security_user_scope":       {"UserUPN": "string", "RoleName": "string", "Region": "string"},
    "admin_inventory_artifacts": {"ArtifactId": "string", "ArtifactType": "string", "ArtifactName": "string",
                                  "Workspace": "string", "OwnerUPN": "string", "LastViewedDate": "date",
                                  "Certified": "boolean", "SizeMB": "double", "LicenseTier": "string"},
    "ops_refresh_history":       {"RefreshTimestamp": "timestamp", "DatasetName": "string", "Workspace": "string",
                                  "Capacity": "string", "Status": "string", "DurationMinutes": "int",
                                  "ErrorCategory": "string"},
    "ops_capacity_metrics_5min": {"Timestamp": "timestamp", "CapacityName": "string", "CPUUtilizationPct": "double",
                                  "MemoryUtilizationPct": "double", "QueryWaitMsP95": "int",
                                  "ThrottlingEvents": "int", "InteractiveQueries": "int"},
}
LEGACY_TABLES = list(SCHEMAS)                                   # each legacy CU has all ten tables
REF_TABLES    = ["glossary_kpi_definitions", "security_user_scope"]

## Helpers

In [ ]:
import io, time, requests, pandas as pd
import sempy.fabric as fabric

assert "yourtenant" not in TENANT_DOMAIN, "Set TENANT_DOMAIN to your tenant's domain first."
WORKSPACE_ID = fabric.get_workspace_id()
rest = fabric.FabricRestClient()

def _props(lh):
    d = lh if isinstance(lh, dict) else getattr(lh, "__dict__", {})
    return d.get("properties") or {}

def ensure_lakehouse(name):
    """Return (abfs_path, schema_enabled, sql_endpoint_id), creating the lakehouse if needed."""
    try:
        lh = notebookutils.lakehouse.get(name)
    except Exception:
        print(f"Creating lakehouse {name} ...")
        notebookutils.lakehouse.create(name)
        lh = notebookutils.lakehouse.get(name)
    p = _props(lh)
    for _ in range(30):                          # the SQL endpoint is provisioned asynchronously
        if p.get("sqlEndpointProperties", {}).get("id"):
            break
        time.sleep(10); p = _props(notebookutils.lakehouse.get(name))
    return p["abfsPath"], bool(p.get("defaultSchema")), p.get("sqlEndpointProperties", {}).get("id")

def read_csv_text(folder, table):
    if DATA_SOURCE == "github":
        url = f"{REPO_RAW_BASE}/{folder}/{table}.csv"
        r = requests.get(url, timeout=60); r.raise_for_status()
        return spark.createDataFrame(pd.read_csv(io.StringIO(r.text), dtype=str, keep_default_na=False))
    ref_abfs, _, _ = ensure_lakehouse(LAKEHOUSES["enterprise_ref"])
    return spark.read.option("header", True).csv(f"{ref_abfs}/Files/data/{folder}/{table}.csv")

def typed(df, table):
    cols = []
    for c, t in SCHEMAS[table].items():
        v = F.when(F.trim(F.col(c)) == "", None).otherwise(F.trim(F.col(c)))
        if t == "boolean":
            v = v.isin("1", "true", "True", "TRUE")
        cols.append(v.cast(t).alias(c))
    return df.select(*cols)

def write_table(df, abfs, schema_enabled, table):
    path = f"{abfs}/Tables/{'dbo/' if schema_enabled else ''}{table}"
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(path)
    return spark.read.format("delta").load(path).count()

def refresh_sql_endpoint(name, sql_id):
    """Ask the SQL analytics endpoint to pick up new tables now instead of waiting for background sync."""
    if not sql_id:
        print(f"  {name}: no SQL endpoint id yet, skipped refresh"); return
    try:
        r = rest.post(f"v1/workspaces/{WORKSPACE_ID}/sqlEndpoints/{sql_id}/refreshMetadata", json={})
        if r.status_code == 202 and "Location" in r.headers:
            loc = r.headers["Location"].split("api.fabric.microsoft.com/")[-1]
            for _ in range(60):
                time.sleep(int(r.headers.get("Retry-After", 5)))
                s = rest.get(loc).json().get("status")
                if s in ("Succeeded", "Failed", "Cancelled"): break
            print(f"  {name}: SQL endpoint refresh {s}")
        else:
            print(f"  {name}: SQL endpoint refresh HTTP {r.status_code}")
    except Exception as e:
        print(f"  {name}: SQL endpoint refresh not available ({e}). Use Refresh in the SQL endpoint if tables are missing.")

## Load the legacy credit unions

In [ ]:
summary = []
if LOAD_LEGACY_SOURCES:
    for folder in ["contoso_cu", "fabrikam_cu"]:
        name = LAKEHOUSES[folder]
        abfs, schema_enabled, sql_id = ensure_lakehouse(name)
        for t in LEGACY_TABLES:
            n = write_table(typed(read_csv_text(folder, t), t), abfs, schema_enabled, t)
            summary.append((name, t, n))
        refresh_sql_endpoint(name, sql_id)
else:
    print("LOAD_LEGACY_SOURCES = False: skipping Contoso_CU and Fabrikam_CU (mirrored databases expected).")

## Load the enterprise reference data (certified glossary + RLS scope)

In [ ]:
name = LAKEHOUSES["enterprise_ref"]
abfs, schema_enabled, sql_id = ensure_lakehouse(name)

gloss = typed(read_csv_text("enterprise_ref", "glossary_kpi_definitions"), "glossary_kpi_definitions")
summary.append((name, "glossary_kpi_definitions", write_table(gloss, abfs, schema_enabled, "glossary_kpi_definitions")))

scope = typed(read_csv_text("enterprise_ref", "security_user_scope"), "security_user_scope") \
          .withColumn("UserUPN", F.lower(F.regexp_replace("UserUPN", r"\{tenant\}", TENANT_DOMAIN)))
summary.append((name, "security_user_scope", write_table(scope, abfs, schema_enabled, "security_user_scope")))
refresh_sql_endpoint(name, sql_id)
display(scope)

## Summary

In [ ]:
display(spark.createDataFrame(summary, "lakehouse string, table string, rows long"))
print("Next: in the Combined-Data warehouse run sql/warehouse/01_create_schema_and_tables.sql (first time only), "
      "02_usp_Load_Combined.sql, then EXEC [Combine-Data].[usp_Load_Combined];")